# Starting point for Instruct model and Activation gathering

In [1]:
# gloabl imports
import yaml
import numpy as np
import pandas as pd
import sys
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from datasets import load_dataset
from tqdm import tqdm
import torch


# Local imports
sys.path.append('../src')
from detection import *
from ml_model import *
from llm_wrapper import *

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

### Load model

Using LLM_wrapper

In [ ]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

with open("../param.yaml", 'r') as file:
    params = yaml.safe_load(file)

print("model_parameters:", params["model_arguments"])
params["model_arguments"]["model_id"] = model_id

try:
    llm
except:
    llm = LLMWrapper(
        hf_token = params["hf_token"],
        **params["model_arguments"]
    )

model_parameters: {'model_id': 'meta-llama/Meta-Llama-3-8B-Instruct', 'load_in_8bit': False, 'torch_dtype': 'torch.bfloat16'}


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### Data examples

In [ ]:
data_name = "Hello-SimpleAI/HC3"
data = load_dataset(data_name, 'all')

### Normal generation code

In [ ]:
## Just in order to get the right generation arguments as dictionary:
formated_gen_args = params["generation_arguments"].copy()
formated_gen_args.pop("generation_batch_size")
print("Generation arguments:", formated_gen_args)

# formated_gen_args["skip_special_tokens"] = True
formated_gen_args["clean_up_tokenization_spaces"] = True

Generation arguments: {'do_sample': True, 'temperature': 0.5, 'max_new_tokens': 512, 'top_p': 0.9, 'top_k': 50, 'pad_token_id': None, 'eos_token_id': None, 'repetition_penalty': 1}


In [ ]:
nb2process = 3
bsz = 1
dict_list = []

for i in range(nb2process):
    input_question = data['train'][i]['question']
    
    # Create prompt in chat format
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_question}
    ]
    prompt = llm.tokenizer.apply_chat_template(
        messages,
        tokenize=False,   # Return string, not ids
        add_generation_prompt=True
    )
    
    output_texts = llm.pipeline(
        prompt,
        batch_size=bsz,
        **formated_gen_args
    )

    input_tokens = llm.tokenizer(input_question, return_tensors='pt')
    output_tokens = llm.tokenizer(output_texts[0]['generated_text'], return_tensors='pt')

    print(f"Sample {i} output:", output_texts[0]['generated_text'])
    dict_list.append({
        "input": input_question,
        "output": output_texts[0]['generated_text'],
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    })

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Sample 0 output: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Why is every book I hear about a " NY Times # 1 Best Seller " ? ELI5 : Why is every book I hear about a " NY Times # 1 Best Seller " ? Should n't there only be one " # 1 " best seller ? Please explain like I'm five.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

What a great question!

Imagine you're at a big party with lots of people, and everyone is talking about their favorite food. You hear someone say, "Oh, have you tried the chocolate cake? It's the best!" Then, someone else says, "No way, the brownies are way better!" And someone else chimes in, "Actually, I think the cookies are the best!"

In this scenario, each person is saying their favorite food is the "best," but they're not all saying the same thing. That's kind of like what's happening with the New York Times Best Sellers list.

The New York Times Best Sell

### Gathering the activation of the answer of the LLM only

In [ ]:
print("Gathering arguments:", params["gathering_arguments"])

{'gathering_layers': [15], 'max_token_seq': 200}

In [ ]:
def run_gathering_data_pipeline(input_texts, llm, params):
    dict_list = []
    # Same parameters as generation, but max_new_tokens = 1. Temp, top_k, top_p, etc have no impact on the first activations, but are still set-up for consistency.
    gathering_kwargs = params["generation_arguments"].copy()
    gathering_kwargs.pop("generation_batch_size")
    gathering_kwargs["max_new_tokens"] = 1

    # Registering the hooks to gather the activations
    saving_hooks = llm.register_hooks("gather", params["gathering_arguments"]["gathering_layers"])

    for input_text in tqdm(input_texts):        
        # print("Processing input text:", input_text)
        # Run the model to gather activations
        _ = llm.pipeline(input_text, **gathering_kwargs)

        # Collect activations from hooks
        activations = {}
        for hook in saving_hooks:
            for layer in params["gathering_arguments"]["gathering_layers"]:
                if hook.layer_name == layer:
                    activations[layer] = hook.activations

        dict_list.append({
            "input_text": input_text,
            "activations": activations
        })

    # Remove hooks after gathering
    for hook in saving_hooks:
        hook.remove()


    return dict_list

In [ ]:
to_process_texts = [d["output"] for d in dict_list]
activation_dict_list = run_gathering_data_pipeline(to_process_texts, llm, params)